In [118]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from ultralytics import SAM, FastSAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision import transforms
from torchvision.models import resnet50
import torch
import numpy as np
from PIL import Image, ImageOps
import cv2
import os
import random
import json, urllib.request
from torch.nn import functional as F

In [28]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FOLDER = "C:/! Photos/album 2021"
EXTS   = (".jpg", ".jpeg", ".png")
files = []
for root, _, fnames in os.walk(FOLDER):
    for f in fnames:
        if f.lower().endswith(EXTS):
            files.append(os.path.join(root, f))
if not files:
    raise RuntimeError(f"Aucune image {EXTS} trouvée dans '{FOLDER}'")

In [265]:
img_name = random.choice(files)
img_path = os.path.join(FOLDER, img_name)
#img_path = "images/IMG_4104.JPG"
print(f"Image choisie : {img_name}")


img_pil  = Image.open(img_path).convert("RGB")
img_pil = ImageOps.exif_transpose(img_pil)
h, w = img_pil.size

img_pil = img_pil.resize((h//10, w//10))
print(h//10, w//10)


transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225]) 
])
input_tensor = transform(img_pil).unsqueeze(0).to(device)

rgb_img = np.array(img_pil).astype(np.float32) / 255.0 

Image choisie : C:/! Photos/album 2021\2023\Turquie 2023\Ped\P1100798.JPG
377 252


In [124]:
model = resnet50(pretrained=True).to(device).eval()
target_layers = [model.layer4[-1]]
targets = None
model_sam = FastSAM("FastSAM-s.pt").to(device)

In [266]:
with GradCAM(model=model, target_layers=target_layers) as cam:
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)


with GradCAMPlusPlus(model=model, target_layers=target_layers) as camplusplus:
    grayscale_camplusplus = camplusplus(input_tensor=input_tensor, targets=targets)[0]

visualizationplusplus = show_cam_on_image(rgb_img, grayscale_camplusplus, use_rgb=True)


results_sam = model_sam(np.array(img_pil))
masks = results_sam[0].masks.data.cpu().numpy()
N, H, W = masks.shape
palette = np.random.default_rng(42)
palette = palette.integers(0, 256, size=(len(masks), 3), dtype=np.uint8)

visualization_sam = np.zeros((H, W, 3), dtype=np.uint8)

for i, mask in enumerate(masks):
    color = palette[i]
    visualization_sam[mask > 0] = color


url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = urllib.request.urlopen(url).read().decode().splitlines()


with torch.no_grad():
    logits = model(input_tensor) 
probas = F.softmax(logits, dim=1)[0]

top5 = torch.topk(probas, 5)
for rank, (idx, p) in enumerate(zip(top5.indices, top5.values), 1):
    print(f"{rank}. {labels[idx]}  (index={idx}, p={p:.2%})")

cv2.imshow("Grad-CAM", cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR))
cv2.imshow("Grad-CAMPlusPlus", cv2.cvtColor(visualizationplusplus, cv2.COLOR_RGB2BGR))
cv2.imshow("SAM", cv2.cvtColor(visualization_sam, cv2.COLOR_RGB2BGR))
cv2.waitKey(0)
cv2.destroyAllWindows()


0: 448x640 52 objects, 34.1ms
Speed: 1.0ms preprocess, 34.1ms inference, 4.0ms postprocess per image at shape (1, 3, 448, 640)
1. sandbar  (index=977, p=24.84%)
2. swimming trunks  (index=842, p=20.68%)
3. alp  (index=970, p=13.73%)
4. cliff  (index=972, p=11.17%)
5. geyser  (index=974, p=5.15%)


# Analyse de quelques résultats :
J'ai utilisé comme modèle de classification ResNet50, puis ai appliqué grad-cam et grad-camplusplus. Enfin, je générais les masques de FastSAM. Comparons tous ça sur quelques exemples.


![image1](images/Capture%20d'écran%202025-06-28%20112559.png)

On remarque ici que la 3ème classe prédite est la bonne (dinning table), bien que restaurant (2ème) pourrait aussi convenir. Mais globalement le modèle n'est pas sûr de lui. Et la localisation de grad-cam semble plutôt bonne, grad-cam++ n'apportant pas grand chose. SAM arrive à extraire un bon nombre de composant dans l'image, bien que certains ne soient pas stable, comme les lunette détectée sur la 1ère personne mais pas sur la 3ème. Ou encore, la 4ème personne qui n'a ni chevelure ni visage.

![image1](images/Capture%20d'écran%202025-06-28%20112656.png)

On remarque que là les 2 premières prédictions sont possibles, et que c'est bien localisé sur le chateau. SAM arrive à extraire le chateau en trois blocs (+ 3 bouts de tours), en excluant bien les arbres devant, ce qui n'est pas parfait mais tout de même assez bon.

![image1](images/Capture%20d'écran%202025-06-28%20113120.png)

On remarque que le resnet reconnaît bien la falaise, bien que son attention ne soit que sur une partie d'elle. Sinon pour SAM, on remarque la différence de traitement entre la personne proche qui a plusieurs détails, et la personne loin qui ne devient qu'un seul bloc. Il ne devrait pas détecter la falaise car c'est le fond de l'image, mais pourtant il y a quelques petit bloc par-ci par-là, avec également deux gros blocs au-dessus et en haut à droite. Celui en haut à droite peut s'expliquer car c'est un autre flan de la falaise, mais celui du dessus moins, car il n'y a pas de coupure entre elle et le reste de cette partie de la falaise. Pour finir, SAM arrive à identifier certaines accroches de la via ferrata, mais pas toutes. Encore une fois, pas de différence notable entre grad-cam et grad-cam++.

![image1](images/Capture%20d'écran%202025-06-28%20113150.png)

Ici Resnet détecte bien la tente de montagne, qui est au centre de cailloux. La localisation englobe toute la tente. Sam arrive également à l'extraire, mais en 2 blocs, dû au fait que la tente à deux parties avec à chaque fois une couleur différente. Fait notable, SAM extrait les plus gros cailloux, car ils ressortent du sol. Une partie du ciel forme aussi un bloc, mais cela est moins expliquable car tout le seul n'est pas pris dedans.

![image1](images/Capture%20d'écran%202025-06-28%20113623.png)

Ici on n'a que du paysage. SAM parvient à presque tout extraire. Les bâtiments du village ont leur propre bloc (maisons isolées), la chaine de montagne en face en forme un unique, et sinon les autres blocs semblent assez cohérents. Pour ce qui est de resnet, on pourrait qualifier les deux premières classifications comme bonnes, puisqu'on ne peut pas vraiment différencier un alpage d'une vallée ici. La concentration mise en évidence par grad-cam montre un intérêt sur une grande partie de l'image, ce qui est assez inhabituel.

![image1](images/Capture%20d'écran%202025-06-28%20113855.png)

Enfin, on a ici une image où resnet et SAM ont beaucoup de difficultés. Déjà, la classification n'a aucun sens (saumon en premier). Ensuite, on remarque bien qu'il ne sait pas trop où regarder, vu la différence entre gard-cam et grad-cam++ (une telle différence est anormale). Pour ce qui est de SAM, la forme des blocs semble aléatoire, et leurs frontières n'ont pas trop de liens avec l'image. Tout ceci est assez étrange, puisque l'image ne semble pas être bien différente des précédentes.


# Difficultés :
Ma plus grosse difficulté a été mon espace de stockage, mais ce n'est pas directement lié à ce travail. Sinon, j'ai eu quelques difficultés pratiques. J'ai mis du temps à me rendre compte que SAM donnait la même couleur à tous les masques, ce que j'interprétais comme étant un gros masque unique. Cela change absolument tout, et j'ai dû regénérer mes captures d'écrans. J'ai également eu quelques problèmes avec l'installation de la librairie grad-CAM (importée ici sous le nom pytorch_grad_cam). Sinon, je n'ai pas eu trop de problème sur le code en lui-même, hormis la gestion de la taille des images, qui ne devait pas être trop grandes (choix de diviser par 10 la hauteur et largeur, pour avoir en majorité des images en 400 par 300 ou inversement), ainsi que l'orientation des images (entre portrait et paysage).